In [1]:
import pandas as pd
import numpy as np

In [10]:
df = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\results_clean.csv")
df.head()


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight,result
0,1990-01-12,Algeria,Mali,5,0,Friendly,Paris,France,True,0.5,H
1,1990-01-14,Algeria,Cameroon,3,1,Friendly,Paris,France,True,0.5,H
2,1990-01-17,Greece,Belgium,2,0,Friendly,Athens,Greece,False,0.5,H
3,1990-01-17,Mexico,Argentina,2,0,Friendly,Los Angeles,United States,True,0.5,H
4,1990-01-20,Malawi,Tanzania,2,2,Friendly,Lobamba,Swaziland,True,0.5,D


In [11]:
# Starting rating for every team
STARTING_ELO = 1500

# K-factor — controls how much ratings change after each match
# We multiply by tournament_weight so WC matches matter more
BASE_K = 30

# Home advantage — home team gets a bonus
HOME_ADVANTAGE = 100

# Dictionary to store every team's current rating
elo_ratings = {}

def get_elo(team):
    if team not in elo_ratings:
        elo_ratings[team] = STARTING_ELO
    return elo_ratings[team]

print("Elo system ready!")

Elo system ready!


In [12]:
def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

def update_elo(home_team, away_team, home_score, away_score, tournament_weight, neutral):
    # Get current ratings
    home_elo = get_elo(home_team)
    away_elo = get_elo(away_team)
    
    # Add home advantage if not a neutral venue
    if neutral:
        home_elo_adjusted = home_elo
    else:
        home_elo_adjusted = home_elo + HOME_ADVANTAGE
    
    # Expected scores
    home_expected = expected_score(home_elo_adjusted, away_elo)
    away_expected = expected_score(away_elo, home_elo_adjusted)
    
    # Actual scores
    if home_score > away_score:
        home_actual, away_actual = 1, 0
    elif home_score < away_score:
        home_actual, away_actual = 0, 1
    else:
        home_actual, away_actual = 0.5, 0.5
    
    # K factor adjusted by tournament weight
    k = BASE_K * tournament_weight
    
    # Update ratings
    elo_ratings[home_team] = home_elo + k * (home_actual - home_expected)
    elo_ratings[away_team] = away_elo + k * (away_actual - away_expected)

print("Functions ready!")

Functions ready!


In [13]:
# Sort by date so we process matches in order
df = df.sort_values('date').reset_index(drop=True)

# Loop through every match and update ratings
for _, row in df.iterrows():
    update_elo(
        home_team=row['home_team'],
        away_team=row['away_team'],
        home_score=row['home_score'],
        away_score=row['away_score'],
        tournament_weight=row['tournament_weight'],
        neutral=row['neutral']
    )

print(f"Processed {len(df)} matches!")
print(f"Total teams rated: {len(elo_ratings)}")

Processed 32101 matches!
Total teams rated: 323


In [7]:
elo_df = pd.DataFrame(list(elo_ratings.items()), columns=['team', 'elo'])
elo_df = elo_df.sort_values('elo', ascending=False).reset_index(drop=True)
elo_df.head(20)


,team,elo
0,Spain,2188.176132
1,Argentina,2100.244381
2,France,2074.344160
3,England,2035.785502
4,Ecuador,2024.587277
5,Turkey,2015.297093
6,Brazil,1970.323402
7,Portugal,1970.044524
8,Norway,1968.683891
9,Colombia,1968.493194


In [8]:
STARTING_ELO = 1500
BASE_K = 30
HOME_ADVANTAGE = 100
DECAY = 0.99  # ratings drift 1% back toward 1500 each match

elo_ratings = {}

def get_elo(team):
    if team not in elo_ratings:
        elo_ratings[team] = STARTING_ELO
    return elo_ratings[team]

def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

def update_elo(home_team, away_team, home_score, away_score, tournament_weight, neutral):
    home_elo = get_elo(home_team)
    away_elo = get_elo(away_team)

    # Apply decay — ratings drift toward 1500 over time
    elo_ratings[home_team] = STARTING_ELO + (home_elo - STARTING_ELO) * DECAY
    elo_ratings[away_team] = STARTING_ELO + (away_elo - STARTING_ELO) * DECAY

    home_elo = elo_ratings[home_team]
    away_elo = elo_ratings[away_team]

    if neutral:
        home_elo_adjusted = home_elo
    else:
        home_elo_adjusted = home_elo + HOME_ADVANTAGE

    home_expected = expected_score(home_elo_adjusted, away_elo)
    away_expected = expected_score(away_elo, home_elo_adjusted)

    if home_score > away_score:
        home_actual, away_actual = 1, 0
    elif home_score < away_score:
        home_actual, away_actual = 0, 1
    else:
        home_actual, away_actual = 0.5, 0.5

    k = BASE_K * tournament_weight

    elo_ratings[home_team] = elo_ratings[home_team] + k * (home_actual - home_expected)
    elo_ratings[away_team] = elo_ratings[away_team] + k * (away_actual - away_expected)

# Reprocess all matches
for _, row in df.iterrows():
    update_elo(
        home_team=row['home_team'],
        away_team=row['away_team'],
        home_score=row['home_score'],
        away_score=row['away_score'],
        tournament_weight=row['tournament_weight'],
        neutral=row['neutral']
    )

# View top 20
elo_df = pd.DataFrame(list(elo_ratings.items()), columns=['team', 'elo'])
elo_df = elo_df.sort_values('elo', ascending=False).reset_index(drop=True)
elo_df.index += 1
elo_df.head(20)

,team,elo
1,Spain,1917.311952
2,France,1828.266029
3,Morocco,1827.315061
4,Turkey,1823.420350
5,Argentina,1817.413882
6,England,1817.402483
7,Senegal,1810.813940
8,Nigeria,1804.828657
9,Norway,1785.151106
10,Japan,1784.196888


In [16]:
STARTING_ELO = 1500
HOME_ADVANTAGE = 100

elo_ratings = {}

def get_elo(team):
    if team not in elo_ratings:
        elo_ratings[team] = STARTING_ELO
    return elo_ratings[team]

def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

def update_elo(home_team, away_team, home_score, away_score, tournament_weight, neutral, date):
    home_elo = get_elo(home_team)
    away_elo = get_elo(away_team)

    if neutral:
        home_elo_adjusted = home_elo
    else:
        home_elo_adjusted = home_elo + HOME_ADVANTAGE

    home_expected = expected_score(home_elo_adjusted, away_elo)
    away_expected = expected_score(away_elo, home_elo_adjusted)

    if home_score > away_score:
        home_actual, away_actual = 1, 0
    elif home_score < away_score:
        home_actual, away_actual = 0, 1
    else:
        home_actual, away_actual = 0.5, 0.5

    year = pd.to_datetime(date).year
    if year >= 2022:
        base_k = 35
    elif year >= 2018:
        base_k = 30
    else:
        base_k = 20

    k = base_k * tournament_weight

    elo_ratings[home_team] = home_elo + k * (home_actual - home_expected)
    elo_ratings[away_team] = away_elo + k * (away_actual - away_expected)

for _, row in df.iterrows():
    update_elo(
        home_team=row['home_team'],
        away_team=row['away_team'],
        home_score=row['home_score'],
        away_score=row['away_score'],
        tournament_weight=row['tournament_weight'],
        neutral=row['neutral'],
        date=row['date']
    )

elo_df = pd.DataFrame(list(elo_ratings.items()), columns=['team', 'elo'])
elo_df = elo_df.sort_values('elo', ascending=False).reset_index(drop=True)
elo_df.index += 1
elo_df.head(20)

,team,elo
1,Spain,2169.690841
2,Argentina,2047.584193
3,England,2033.353422
4,France,2020.625388
5,Ecuador,1996.118091
6,Turkey,1993.245154
7,Norway,1956.731310
8,Morocco,1949.295775
9,Colombia,1926.877556
10,Senegal,1925.739655


In [17]:
teams_to_check = ['Germany', 'Portugal', 'Netherlands', 'Italy', 
                   'Belgium', 'Croatia', 'Uruguay', 'USA', 
                   'Ecuador', 'Turkey', 'Norway', 'Paraguay']

check = elo_df[elo_df['team'].isin(teams_to_check)]
print(check.to_string())

           team          elo
5       Ecuador  1996.118091
6        Turkey  1993.245154
7        Norway  1956.731310
11     Paraguay  1923.215588
17     Portugal  1908.832929
18  Netherlands  1895.359242
19      Croatia  1884.979037
20      Germany  1883.699925
22      Uruguay  1875.887717
33        Italy  1810.958705
36      Belgium  1780.811104


In [18]:
gs = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\group_stages_clean.csv")
wc_teams = gs['nation'].unique()

wc_elo = elo_df[elo_df['team'].isin(wc_teams)].reset_index(drop=True)
wc_elo.index += 1
print(f"Total WC teams: {len(wc_elo)}")
wc_elo

Total WC teams: 48


,team,elo
1,Spain,2169.690841
2,Argentina,2047.584193
3,England,2033.353422
4,France,2020.625388
5,Ecuador,1996.118091
6,Turkey,1993.245154
7,Norway,1956.731310
8,Morocco,1949.295775
9,Colombia,1926.877556
10,Senegal,1925.739655


In [19]:
print(gs[gs['nation'] == 'Norway'])

   group  position  nation
35     I         4  Norway


In [20]:
wc_elo.to_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\elo_ratings.csv", index=False)
print("Saved!")

Saved!


In [21]:
# Save ALL teams, not just WC teams
full_elo = pd.DataFrame(list(elo_ratings.items()), columns=['team', 'elo'])
full_elo = full_elo.sort_values('elo', ascending=False).reset_index(drop=True)
full_elo.to_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\elo_all_teams.csv", index=False)
print(f"Saved {len(full_elo)} teams!")

Saved 323 teams!
